# Fundamentals 08 - System API

Objetivo: usar `toolkit.system` como owner de runtime, registry de Tools, Skills, Agents e inspeccion estatica.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_DEMO_SYMBOL | system | Entrada del usuario para el agente registrado. |
| runtime | python-runtime | Mantener la demostracion reproducible. |
| inspect | sin ejecucion | Validar el registro y sus efectos laterales. |

In [ ]:
import os

import agentic_systems as toolkit

SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "system")
runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

## 1) Registrar Tool en el System

In [ ]:
@system.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.PUBLIC_API}

toolkit.show_json({
    "public_tool_names": list(system.public_tool_names),
    "runtime_tool_names": list(system.tool_names),
}, title="System registry")

## 2) Registrar Skill reusable

In [ ]:
inspection_skill = toolkit.skill(
    name="system_public_api_inspection",
    description="Capacidad registrada por el System.",
    tools=[system.public_tools["inspect_public_api"]],
    prompts={"instructions": "Inspecciona simbolos con evidencia."},
    contracts={"default": toolkit.AgentContract(must_call=["inspect_public_api"]).model_dump(mode="json")},
)
system.skill(inspection_skill)
toolkit.show_json({
    "skill_names": list(system.skill_names),
    "runtime_skills": [skill.info() for skill in system.runtime_skills],
}, title="System skills")

## 3) Crear Agent ligado al System

In [ ]:
agent = system.agent(
    name="system_public_api_agent",
    instructions=inspection_skill.instructions,
    skills=[inspection_skill],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": SYMBOL}},
    mode="eval",
)
toolkit.human_result(result, title="System Agent RunResult", show_lineage=True)

## 4) Inspeccionar sin ejecutar componentes

`system.inspect()` es estatico. `toolkit.show` selecciona la explicacion humana de `InspectReport`.

In [ ]:
inspection = system.inspect()
inspection.raise_if_errors()
toolkit.show_json(inspection.to_dict(), title="System inspection structured")
toolkit.show(inspection, title="System inspection human")

assert inspection["side_effects"] == {"models_executed": 0, "tools_executed": 0}

## 5) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.system", "system.tool", "toolkit.skill", "system.skill",
    "system.agent", "agent.run", "toolkit.human_result", "system.inspect",
    "InspectReport.to_dict", "toolkit.show", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="System API coverage")

## Resultado esperado

Registry, Skill y Agent comparten el mismo System. La inspeccion reporta cero side effects propios.